# Explore
Explore what subreddits relate to or mention resumes.

In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv()

True

In [4]:
client_id = os.environ.get("REDDIT_CLIENT_ID")
client_secret = os.environ.get("REDDIT_CLIENT_SECRET")

In [5]:
import praw

client= praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent="research/0.1"
)

## Exploring Subreddits
Found that r/resumes is a prominent subreddit

In [6]:
resume_subs = client.subreddits.search_by_name("resume")

for sub in resume_subs:
    print(f"r/{sub.display_name} - {sub.subscribers:,} subscribers")

r/Resume - 130,812 subscribers
r/resumes - 1,328,408 subscribers
r/ResumeExperts - 29,598 subscribers
r/ResumeTips - 18,118 subscribers
r/ResumeCoverLetterTips - 27,848 subscribers
r/resumesindia - 7,321 subscribers
r/resumesupport - 8,767 subscribers
r/ResumeFairies - 7,082 subscribers
r/resumeph - 16 subscribers
r/resumereview - 2,295 subscribers


## Downloading sample of r/resumes Posts
Retrieved all posts with >=1 comment over time period spanning January 1st, 2026 to November 1st, 2026.

In [13]:
import deepsift

In [38]:
from deepsift.scraper import RedditScraper
from deepsift.filters import Filters
from deepsift.models import PostField

filters = Filters(
    post_include_fields=[
        PostField.ID,
        PostField.TITLE,
        PostField.SELFTEXT,
        PostField.SCORE,
        PostField.NUM_COMMENTS,
        PostField.CREATED_UTC,
        PostField.SUBREDDIT,
        PostField.URL
    ],
    post_filter=lambda post: post.num_comments >= 1,
)

scraper = RedditScraper(
    filters=filters,
    post_workers=4,
    shard_days=30,
)

# uncomment to run extraction
# post_ids = scraper.scrape_posts(
#     subreddits=["resumes"],
#     from_dt="2026-01-01",
#     to_dt="2026-11-01",
#     output_path="data/subreddit_resumes_posts.parquet",
# )

# print(f"Scraped {len(post_ids)} curated posts.")

## Finding Relevant Posts in Retrieved Data

In [58]:
import pandas as pd

posts_path = "data/subreddit_resumes_posts.parquet"
posts = pd.read_parquet(posts_path)

In [59]:
posts.shape

(20827, 8)

In [60]:
# validating the time period was respected
posts["created_utc"] = pd.to_datetime(posts["created_utc"], unit="s")

In [61]:
# this validates that it we did get data for the complete time period requested
posts["created_utc"].describe()

count                         20827
mean     2026-05-14 14:14:26.002256
min             2026-01-01 00:29:21
25%      2026-02-25 15:13:13.500000
50%             2026-05-04 21:39:57
75%             2026-07-28 12:10:49
max             2026-10-31 22:55:34
Name: created_utc, dtype: object

In [62]:
# exploring values of selftext
freq_text = pd.DataFrame(posts["selftext"].value_counts())

In [63]:
text_freq = pd.DataFrame(posts["selftext"].value_counts())

In [64]:
clean_posts = posts.drop_duplicates(subset=["selftext"])

In [69]:
clean_posts = clean_posts[(clean_posts["selftext"] != "[removed]") & (clean_posts["selftext"] != "")]

In [72]:
clean_posts["num_comments"].describe()

count    9507.000000
mean        4.592511
std         8.100342
min         1.000000
25%         1.000000
50%         3.000000
75%         5.000000
max       271.000000
Name: num_comments, dtype: float64

In [74]:
clean_posts["score"].describe()

count    9507.000000
mean        2.981172
std        21.132470
min         0.000000
25%         1.000000
50%         1.000000
75%         2.000000
max      1175.000000
Name: score, dtype: float64